# Construcción del Asesor Inteligente de Hipotecas — CaixaBank TFM

> **Autor:** Trabajo de Fin de Máster  
> **Fecha:** Marzo 2026  
> **Stack:** Python · Flask · scikit-learn · Gemini AI · HTML/CSS/JS

Este notebook documenta el proceso completo de construcción de la aplicación web del TFM: desde la replicación del diseño de CaixaBank hasta el despliegue del modelo de predicción de diferenciales hipotecarios con explicabilidad por variable.

---

## Índice

1. [Replicar y adaptar el frontend de CaixaBank](#1)
2. [Web scraping de datos fiscales por CCAA](#2)
3. [Integración del dataset macroeconómico](#3)
4. [Modelo de predicción del diferencial hipotecario](#4)
5. [Sistema de tipos reales APRC/Euribor](#5)
6. [Lógica bancaria: DTI, ahorros, financiación joven por CCAA](#6)
7. [Integración de IA generativa: Gemini + Noa](#7)
8. [Panel de administración](#8)
9. [Arquitectura final de la aplicación](#9)

---
<a id='1'></a>
## 1. Replicar y adaptar el frontend de CaixaBank

### 1.1 Objetivo

El TFM simula un entorno bancario real. Para que la aplicación sea creíble y profesional, replicamos el diseño del portal público de CaixaBank (`caixabank.es`) como punto de partida.

### 1.2 Elementos obtenidos de la web de CaixaBank

Usando las herramientas de desarrollador del navegador (F12 → Inspector), identificamos y extrajimos los siguientes elementos:

| Elemento | Origen |
|---|---|
| Logotipo SVG/GIF | URL pública del CDN de CaixaBank |
| Paleta de colores corporativa | CSS inspeccionado (azul `#003366`, azul claro `#0072c6`, naranja `#f0a500`) |
| Tipografía | `Open Sans` (Google Fonts) — coincide con la fuente corporativa |
| Estructura del header | Three-tier: `.header-top` (segmento cliente), `.header-middle` (logo + acciones), `.header-bottom` (navegación) |
| Botones corporativos | Clase `.button--blue` y `.button--ocher` con bordes redondeados |

El logo se obtiene directamente desde la URL pública:
```
https://www.caixabank.es/deployedfiles/common/R2016/Estaticos/images/gif/logo_laCaixa_principal.gif
```

### 1.3 Adaptaciones realizadas

El diseño replicado se adaptó al contexto del TFM con los siguientes cambios:

- **Navegación simplificada**: Solo se mantienen las secciones relevantes al análisis hipotecario: *Dashboard Analítico*, *Test Viabilidad*, *Hipotecas*.
- **Eliminación de emojis**: La UI adopta un tono 100% profesional, sin emojis en ningún elemento visible.
- **Nuevas secciones**: Se añade el *Dashboard Analítico* con gráficos dinámicos de Chart.js y el *Asesor Inteligente de Hipotecas* (flujo de 3 pasos).
- **Panel de administración**: Ruta `/admin` con login (`caixabank`/`caixabank`) para gestionar el dataset y los tipos de hipotecas.

### 1.4 Estructura de ficheros resultante

```
Web/
├── app.py                    # Backend Flask
├── templates/
│   ├── index.html            # Dashboard analítico
│   ├── hipotecas.html        # Asesor inteligente (3 pasos + Noa)
│   ├── admin.html            # Panel de administración
│   ├── login.html            # Login admin
│   └── test_hipoteca.html    # Simulador legacy de viabilidad
└── static/
    ├── style.css             # Estilos corporativos CaixaBank adaptados
    ├── script.js             # Lógica JS del dashboard
    └── tipos_hipotecas.csv   # CSV de tipos APRC/Euribor/Diferencia
```

In [ ]:
# Verificación: colores y elementos HTML clave del diseño CaixaBank
import re

with open('Web/static/style.css', 'r', encoding='utf-8') as f:
    css = f.read()

# Extraemos los colores definidos en CSS
colors = re.findall(r'(#[0-9a-fA-F]{3,6})', css)
from collections import Counter
top_colors = Counter(colors).most_common(10)
print("Colores más usados en style.css:")
for color, count in top_colors:
    print(f"  {color}  ×{count}")

---
<a id='2'></a>
## 2. Web scraping de datos fiscales por CCAA

### 2.1 Problema a resolver

Para validar si un cliente tiene ahorros suficientes para comprar una vivienda, el asesor hipotecario necesita conocer los **impuestos y gastos de compraventa** reales. Estos varían por:

- Comunidad Autónoma (ITP desde el 4% en País Vasco hasta el 10% en Cataluña, Galicia o Valencia)
- Tipo de vivienda: segunda mano (ITP) vs. nueva (IVA 10% + AJD)
- Perfil del comprador: mayores reducciones para jóvenes (≤35 años en la mayoría de CCAA)

### 2.2 Fuente de datos: financialtools.web.app

La web `https://financialtools.web.app/herramientas/calculadora-hipoteca/` es una calculadora fiscal que agrega la normativa actualizada de todas las CCAA para 2026. Toda la lógica está en un fichero JS público:

```
https://financialtools.web.app/herramientas/calculadora-hipoteca/app.js
```

### 2.3 Proceso de extracción (ver notebook `scraping_finantialtools.ipynb`)

```python
import requests, re, json

# 1. Descarga del JS
resp = requests.get('https://financialtools.web.app/herramientas/calculadora-hipoteca/app.js')
js_source = resp.text

# 2. Extracción del objeto AJD por CCAA con regex
ajd_block = extract_js_object(js_source, 'ajdPorCCAA')
ajd_por_ccaa = {ccaa: float(rate) for ccaa, rate in re.findall(r'"([^"]+)"\s*:\s*([0-9.]+)', ajd_block)}

# 3. Extracción de aranceles notaría/registro
notaria_block = extract_js_object(js_source, 'gastosNotaria')
gastos_notaria = {int(k): int(v) for k, v in re.findall(r'(\d+)\s*:\s*(\d+)', notaria_block)}
```

### 2.4 Datos extraídos e integrados en `app.py`

| Estructura Python | Contenido |
|---|---|
| `ITP_POR_CCAA` | Tipo general ITP por CCAA (4–10%) |
| `ITP_JOVEN` | Tipo reducido para compradores ≤35 años |
| `AJD_POR_CCAA` | AJD sobre hipoteca por CCAA (0.5–1.5%) |
| `gastos_notaria` | Aranceles notariales por tramos de precio |
| `gastos_registro` | Aranceles registrales por tramos de precio |

### 2.5 Función de cálculo

```python
def _calc_taxes(precio, ccaa, age, is_new_build=False):
    notaria, registro = _notaria_registro(precio)
    if is_new_build:
        # IVA (10%) + AJD por CCAA
        total = precio * 10/100 + precio * AJD_POR_CCAA[ccaa]/100 + notaria + registro
    else:
        # ITP: tipo joven si edad ≤ 35
        itp_rate = ITP_JOVEN[ccaa] if age <= 35 else ITP_POR_CCAA[ccaa]
        total = precio * itp_rate/100 + notaria + registro
    return total
```

Los impuestos calculados se **descuentan de los ahorros del cliente** antes de comprobar si puede afrontar la entrada.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Datos extraídos del scraping
ITP_POR_CCAA = {
    'Andalucia': 7.0, 'Aragon': 8.0, 'Asturias': 8.0, 'Baleares': 8.0,
    'Canarias': 6.5, 'Cantabria': 10.0, 'Castilla-La Mancha': 9.0,
    'Castilla y Leon': 8.0, 'Cataluna': 10.0, 'Comunidad Valenciana': 10.0,
    'Extremadura': 8.0, 'Galicia': 10.0, 'Madrid': 6.0, 'Murcia': 8.0,
    'Navarra': 6.0, 'Pais Vasco': 4.0, 'La Rioja': 7.0, 'Ceuta': 6.0, 'Melilla': 6.0
}

fig, ax = plt.subplots(figsize=(14, 4))
ccaas = sorted(ITP_POR_CCAA, key=lambda x: ITP_POR_CCAA[x])
vals = [ITP_POR_CCAA[c] for c in ccaas]
bars = ax.barh(ccaas, vals, color=['#003366' if v < 8 else '#c0392b' if v >= 10 else '#0072c6' for v in vals], alpha=0.85)
ax.axvline(x=8, color='orange', linestyle='--', linewidth=1.2, label='Media estatal (8%)')
ax.set_xlabel('ITP segunda mano (%)')
ax.set_title('ITP por Comunidad Autónoma — vivienda segunda mano 2026', fontweight='bold')
ax.legend()
for bar, val in zip(bars, vals):
    ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2, f'{val}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. Integración del dataset macroeconómico

### 3.1 Dataset principal: `dataset_final_tfm.csv`

El dataset contiene datos trimestrales de **52 provincias españolas** con las siguientes variables:

| Variable | Descripción | Fuente |
|---|---|---|
| `Trimestre` | Periodo (ej. `2020-Q1`) | INE |
| `Provincia` | Nombre de la provincia | INE |
| `Precio_m2` | Precio medio del m² (EUR) | Tinsa / INE |
| `Euribor` | Euribor a 12 meses (%) | BCE / BdE |
| `Spread_Hipotecas` | Diferencial bancario histórico (%) | BdE |
| `IPC` | Índice de Precios al Consumo (%) | INE |
| `Tasa_Paro` | Tasa de paro nacional (%) | INE/EPA |
| `PIB_pc` | PIB per cápita (EUR) | INE |
| `tasa_ahorro` | Tasa de ahorro de los hogares (%) | INE |
| `Hipotecas_Total` | Número de hipotecas concedidas | INE |

### 3.2 Preparación y carga en Flask

El dataset se carga al inicio de la aplicación Flask en un DataFrame global `df`. El panel de administración permite reemplazarlo en caliente sin reiniciar el servidor:

```python
DATASET_PATH = os.path.join(app.config['UPLOAD_FOLDER'], 'dataset_final_tfm.csv')
df = pd.read_csv(DATASET_PATH)

@app.route('/admin', methods=['POST'])
def admin():
    file.save(file_path)
    global df
    df = pd.read_csv(file_path)   # recarga en memoria sin reiniciar
```

### 3.3 API de datos para el dashboard

El endpoint `/api/data?year=<año>` agrega los datos y devuelve JSON con:
- Ranking de precio m² por provincia (top 20) → gráfico de barras
- Serie temporal de Euribor y Spread → gráfico de líneas
- Años disponibles → selector de filtro global

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Cargamos el dataset para exploración
df = pd.read_csv('../dataset_final_tfm.csv')
print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print(f"Trimestres: {df['Trimestre'].nunique()} | Provincias: {df['Provincia'].nunique()}")
print(f"Periodo: {df['Trimestre'].min()} → {df['Trimestre'].max()}")
df.describe().round(2)

In [ ]:
# Matriz de correlación entre variables macroeconómicas
import seaborn as sns

MACRO_COLS = ['Euribor', 'Spread_Hipotecas', 'IPC', 'Tasa_Paro', 'Precio_m2', 'PIB_pc', 'tasa_ahorro', 'Hipotecas_Total']
corr = df[MACRO_COLS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Correlación entre variables macroeconómicas (dataset completo)', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(f"\nCorrelación Euribor ↔ Spread_Hipotecas: {corr.loc['Euribor','Spread_Hipotecas']:.3f}")
print(f"Correlación IPC ↔ Euribor: {corr.loc['IPC','Euribor']:.3f}")

---
<a id='4'></a>
## 4. Modelo de predicción del diferencial hipotecario

### 4.1 Objetivo del modelo

El asesor hipotecario no se limita a mostrar el tipo de interés actual: necesita **explicar** al cliente _por qué_ el banco aplica ese diferencial sobre el Euribor. Para ello usamos un modelo de regresión lineal múltiple sobre el histórico de variables macroeconómicas.

> **Variable objetivo (Y):** `Spread_Hipotecas` — diferencial bancario histórico medido en el dataset maestro
> 
> **Fuente de verdad para el tipo real:** `tipos_hipotecas.csv` (APRC − Euribor)

El modelo tiene un propósito **explicativo**, no predictivo: los coeficientes normalizados nos muestran qué variable empuja más el spread en cada año.

### 4.2 Variables del modelo

| Variable (X) | Descripción | Efecto esperado sobre Spread |
|---|---|---|
| `Euribor` | Tipo de referencia del mercado interbancario | Negativo (cuando Euribor sube, el spread cae) |
| `IPC` | Inflación general | Positivo (mayor inflación → bancos exigen más margen) |
| `Tasa_Paro` | Nivel de desempleo | Positivo (más riesgo de impago → spreads altos) |
| `Precio_m2` | Precio del activo subyacente | Negativo (vivienda más cara → mejor garantía) |
| `PIB_pc` | Riqueza promedio | Negativo (mayor solvencia media → menor risk premium) |
| `tasa_ahorro` | Capacidad de ahorro de los hogares | Negativo (mejora la solvencia del deudor) |

### 4.3 Pipeline de entrenamiento

```python
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

FEATURES = ['Euribor', 'IPC', 'Tasa_Paro', 'Precio_m2', 'PIB_pc', 'tasa_ahorro']
TARGET   = 'Spread_Hipotecas'

# 1. Filtrar filas con datos completos
model_data = df[FEATURES + [TARGET]].dropna()

# 2. Estandarizar para comparar coeficientes entre variables con escalas distintas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(model_data[FEATURES])

# 3. Ajustar regresión lineal
reg = LinearRegression()
reg.fit(X_scaled, model_data[TARGET])

# 4. Para el año N, calcular la contribución de cada variable
year_vals_scaled = scaler.transform([valores_del_año_N])
contributions = reg.coef_ * year_vals_scaled[0]
# → cada elemento indica cuántos puntos porcentuales aporta esa variable al spread
```

### 4.4 Interpretación de la contribución por variable

**Contribución = coeficiente_normalizado × valor_estandarizado**

Esta descomposición aditiva permite mostrar al cliente un gráfico de barras donde cada variable tiene un "peso" en el diferencial, con signo positivo (aumenta el margen bancario) o negativo (lo reduce). Es el principio de los métodos SHAP simplificado para regresión lineal.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import numpy as np

FEATURES = ['Euribor', 'IPC', 'Tasa_Paro', 'Precio_m2', 'PIB_pc', 'tasa_ahorro']
TARGET   = 'Spread_Hipotecas'

model_data = df[FEATURES + [TARGET]].dropna()
print(f"Filas usadas para entrenamiento: {len(model_data):,}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(model_data[FEATURES].values)
y = model_data[TARGET].values

reg = LinearRegression()
reg.fit(X_scaled, y)

# R² en CV de 5 folds
cv_scores = cross_val_score(reg, X_scaled, y, cv=5, scoring='r2')
print(f"R² medio (CV-5): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"R² en train:     {reg.score(X_scaled, y):.4f}")

# Tabla de coeficientes normalizados
coef_df = pd.DataFrame({'Variable': FEATURES, 'Coeficiente (std.)': reg.coef_}).sort_values('Coeficiente (std.)')
print("\nCoeficientes normalizados:")
coef_df

In [ ]:
# Visualización: coeficientes del modelo
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#28a745' if c < 0 else '#c0392b' for c in coef_df['Coeficiente (std.)']]
ax.barh(coef_df['Variable'], coef_df['Coeficiente (std.)'], color=colors, alpha=0.85)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente normalizado (impacto sobre Spread_Hipotecas)')
ax.set_title('Coeficientes del modelo de regresión lineal\n(negativo = reduce el spread, positivo = lo aumenta)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Simulación: contribuciones para un año concreto (2021 — Euribor negativo, spreads bajos)
year = '2021'
ydf = df[df['Trimestre'].str.startswith(year)]
agg = ydf[FEATURES].mean()

year_arr = np.array([agg[f] for f in FEATURES]).reshape(1, -1)
year_scaled = scaler.transform(year_arr)
contributions = reg.coef_ * year_scaled[0]

contribs_df = pd.DataFrame({'Variable': FEATURES, 'Valor': [agg[f] for f in FEATURES], 'Contribucion (%)': contributions})
contribs_df = contribs_df.sort_values('Contribucion (%)', key=abs, ascending=False)

print(f"Contribuciones para el año {year}:")
print(contribs_df.to_string(index=False))
print(f"\nDiferencial estimado por el modelo: {contributions.sum():.3f}%")

In [ ]:
# Gráfico de contribuciones por variable para el año seleccionado (viz idéntica al asesor web)
fig, ax = plt.subplots(figsize=(9, 3.5))
colors = ['#c0392b' if c >= 0 else '#28a745' for c in contribs_df['Contribucion (%)']]
ax.barh(contribs_df['Variable'], contribs_df['Contribucion (%)'], color=colors, alpha=0.85)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Contribución al diferencial (%)')
ax.set_title(f'Descomposición del diferencial hipotecario — Año {year}', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Sistema de tipos reales APRC/Euribor (tipos_hipotecas.csv)

### 5.1 El problema del diferencial calculado vs. el real

El dataset maestro contiene `Spread_Hipotecas`, pero este es un indicador del BCE/BdE que **no coincide exactamente** con el diferencial que cobra un banco en ventanilla. Para aumentar la precisión, incorporamos un segundo CSV con datos de mercado reales:

```
tipos_combinados_hipotecas_trim.csv
```

| Columna | Descripción |
|---|---|
| `TIME_PERIOD` | Trimestre (ej. `2020-Q1`) |
| `APRC_VALUE` | TAE media del mercado hipotecario (Annual Percentage Rate of Charge) |
| `EURIBOR_VALUE` | Euribor a 12 meses |
| `Diferencia` | `APRC − Euribor` = diferencial real banco-Euribor |

### 5.2 Por qué APRC − Euribor es la métrica correcta

La **APRC** (TAE) es la tasa efectiva que paga el cliente, ya recogida en los datos del BCE. El Euribor es el coste de financiación del banco en el mercado interbancario. La diferencia `APRC − Euribor` es, por tanto, el **margen neto real** que aplica el sistema bancario, incluyendo:
- Prima de riesgo crediticio
- Margen de beneficio bancario
- Costes operativos implícitos

> **Ejemplo 2023:** Euribor = 3.86%, APRC = 3.89% → Diferencial = **0.03%** (mínimo histórico por la convergencia de tipos)
> 
> **Ejemplo 2013:** Euribor = 0.54%, APRC = 3.19% → Diferencial = **2.66%** (máximo spread post-crisis)

### 5.3 Cómo se usa en la aplicación

```python
TIPOS_PATH = 'static/tipos_hipotecas.csv'
df_tipos = pd.read_csv(TIPOS_PATH)
df_tipos['year'] = df_tipos['TIME_PERIOD'].str[:4]

def _get_tipos_for_year(year_str):
    ydata = df_tipos[df_tipos['year'] == str(year_str)]
    return {
        'aprc':      float(ydata['APRC_VALUE'].mean()),
        'euribor':   float(ydata['EURIBOR_VALUE'].mean()),
        'diferencia': float(ydata['Diferencia'].mean()),  # ← tipo real banco
    }

# Tipo final ofrecido en hipoteca variable:
rate = euribor + diferencia_final   # diferencia_final = Diferencia ± ajuste_por_perfil
```

In [ ]:
df_tipos = pd.read_csv('../tipos_combinados_hipotecas_trim.csv')
df_tipos['year'] = df_tipos['TIME_PERIOD'].str[:4]
anual = df_tipos.groupby('year')[['APRC_VALUE', 'EURIBOR_VALUE', 'Diferencia']].mean().round(3)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(anual.index, anual['EURIBOR_VALUE'], label='Euribor 12M (%)', color='#0072c6', linewidth=2)
ax.plot(anual.index, anual['Diferencia'],   label='Diferencial real APRC-Euribor (%)', color='#c0392b', linewidth=2)
ax.fill_between(anual.index, anual['EURIBOR_VALUE'], alpha=0.08, color='#0072c6')
ax.set_title('Euribor vs. Diferencial Real de Mercado (APRC − Euribor) 2005-2025', fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('%')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(anual.index[::2])
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print(anual.to_string())

---
<a id='6'></a>
## 6. Lógica bancaria real: DTI, ahorros, financiación joven por CCAA

### 6.1 DTI — Ratio Deuda / Ingresos (3 niveles)

El DTI (Debt-to-Income ratio) es el indicador principal de solvencia para la concesión de hipotecas en España:

```
DTI (%) = (Cuota hipotecaria + Otras deudas fijas) / Ingresos netos × 100
```

La lógica del asesor implementa tres niveles según la práctica habitual de la banca española:

| DTI | Decision | Color |
|---|---|---|
| < 35% | **Aprobada** | Verde |
| 35–45% | **Pendiente de analista de riesgos** | Naranja |
| > 45% | **Denegada** | Rojo |

El campo **"Otras deudas fijas mensuales"** permite incluir préstamos de coche, tarjetas de crédito con cuota, otras hipotecas, etc., para un cálculo más realista.

### 6.2 Verificación de ahorros netos

```python
total_taxes = _calc_taxes(precio, ccaa, age)
effective_savings = savings - total_taxes          # Ahorros después de pagar impuestos
min_down_payment  = precio * (1 - max_financing_pct / 100)
savings_ok        = effective_savings >= min_down_payment
```

### 6.3 Financiación máxima por CCAA — Programa de Cliente Joven

Los programas autonómicos de aval y primera vivienda permiten superar el límite estándar del 80% de CaixaBank. El backend mapea cada provincia a su CCAA y aplica la tabla de financiación máxima:

| CCAA | Financiación máx. | Límite de edad |
|---|---|---|
| Madrid | 95% | ≤ 40 años |
| País Vasco | 100% | ≤ 40 años |
| Cataluña | 95% | ≤ 35 años |
| Murcia | 100% | ≤ 35 años |
| Galicia | 100% | ≤ 40 años |
| *Resto* | 90–95% | ≤ 35 años |

Cuando el cliente cumple los requisitos de edad y CCAA, el asesor muestra el badge:

> *"Cliente joven en [CCAA]: puede financiar hasta el X% del precio de la vivienda (lím. edad: N años)."*

### 6.4 Precio de compra: mercado vs. propio

El asesor ofrece dos modos:
- **Precio medio de mercado**: `Precio_m2 (provincia, año) × 80 m²` calculado del dataset
- **Precio propio**: el usuario introduce el precio real de la vivienda que quiere comprar, ignorando el m²

In [ ]:
# Simulación del flujo de decisión DTI para distintos perfiles

def simular_dti(income, otras_deudas, loan_amount, rate_pct, years=25):
    r = rate_pct / 100 / 12
    n = years * 12
    cuota = loan_amount * r * (1 + r)**n / ((1 + r)**n - 1) if r > 0 else loan_amount / n
    total_oblig = cuota + otras_deudas
    dti = total_oblig / income * 100
    decision = 'APROBADA' if dti < 35 else ('ANALISTA' if dti <= 45 else 'DENEGADA')
    return round(cuota, 2), round(dti, 1), decision

casos = [
    ('Perfil solvente',    4000, 0,   180000, 3.84),
    ('Perfil medio',       2500, 300, 160000, 3.84),
    ('Zona analista',      2000, 400, 150000, 3.84),
    ('Denegado DTI alto',  1500, 500, 160000, 3.84),
]

print(f"{'Perfil':<22} {'Income':>8} {'Deudas':>8} {'Cuota':>8} {'DTI':>7} {'Decision':<12}")
print('─' * 75)
for nombre, income, deudas, loan, rate in casos:
    cuota, dti, dec = simular_dti(income, deudas, loan, rate)
    print(f"{nombre:<22} {income:>8,} {deudas:>8,} {cuota:>8,.2f} {dti:>6.1f}% {dec:<12}")

---
<a id='7'></a>
## 7. Integración de IA Generativa — Gemini 2.5 Flash + Asistente Noa

### 7.1 Dos capas de IA en la aplicación

La aplicación incorpora dos chatbots con propósitos distintos, ambos basados en **Gemini 2.5 Flash** (Google DeepMind):

| Chatbot | Ruta API | Contexto | Propósito |
|---|---|---|---|
| **Asistente AI (TFM)** | `/api/chat` | Dataset completo | Responde preguntas sobre el mercado inmobiliario (Euribor histórico, precios por provincia, etc.) |
| **Noa** | `/api/noa-chat` | Perfil completo del cliente | Asesora al cliente sobre **su hipoteca concreta** tras recibir el resultado |

### 7.2 Infraestructura de conexión

```python
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()                  # Lee GEMINI_API_KEY desde .env
genai.configure(api_key=os.environ['GEMINI_API_KEY'])
gemini_model = genai.GenerativeModel('gemini-2.5-flash')
```

### 7.3 Noa: prompt contextualizado

Lo que hace a Noa especial es que el frontend envía el **contexto completo del cliente** en cada petición:

```python
@app.route('/api/noa-chat', methods=['POST'])
def noa_chat():
    ctx = request.json['context']   # contiene: year, tipo, income, savings, age, province,
                                    # euribor, aprc, rate, dti_ratio, taxes, status...
    prompt = f"""
    Eres Noa, asesora hipotecaria de CaixaBank.
    Ingresos: {ctx['income']:,}€/mes | Ahorros netos: {ctx['effective_savings']:,}€
    Tipo ofrecido: {ctx['rate']:.2f}% TIN | DTI: {ctx['dti_ratio']:.1f}%
    Decision: {'Aprobada' if ctx['approved'] else 'Denegada'}
    ...
    Pregunta: "{user_msg}"
    """
    response = gemini_model.generate_content(prompt)
```

Esto permite que Noa responda con precisión preguntas como:
- *"¿Cuánto me falta para que me aprueben?"*
- *"¿Por qué el diferencial es tan alto?"*
- *"¿Qué pasa si subo el plazo a 30 años?"*

### 7.4 API key y seguridad

La clave se carga desde un fichero `.env` (no versionado en git):

```
# .env
GEMINI_API_KEY=AIzaSy...
```

In [ ]:
# Verificación de conectividad con Gemini (requiere GEMINI_API_KEY en .env)
import os, sys

sys.path.insert(0, 'Web')
from dotenv import load_dotenv
load_dotenv('Web/.env')

api_key = os.environ.get('GEMINI_API_KEY')
print(f"API Key configurada: {'Si (' + api_key[:8] + '...)' if api_key else 'NO — revisar .env'}")

if api_key:
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.5-flash')
    r = model.generate_content("Di 'OK' si recibes este mensaje correctamente. Solo 'OK', nada mas.")
    print(f"Respuesta de Gemini: {r.text.strip()}")

---
<a id='8'></a>
## 8. Panel de Administración

### 8.1 Rutas y funcionalidades

El panel de administración (`/admin`) está protegido con autenticación por sesión Flask:

- **Credenciales:** usuario `caixabank` / contraseña `caixabank`

Permite cuatro acciones desde la interfaz web sin tocar código:

| Acción | Endpoint | Descripción |
|---|---|---|
| Subir dataset | `POST /admin` (`action=dataset`) | Reemplaza `dataset_final_tfm.csv` y recarga `df` en memoria |
| Subir tipos CSV | `POST /admin` (`action=tipos`) | Reemplaza `tipos_hipotecas.csv` y recarga `df_tipos` |
| Publicar gráfico | `POST /admin/upload-chart` | Ejecuta un `.ipynb` o `.py`, extrae los plots matplotlib y los publica en el dashboard |
| Eliminar gráfico | `DELETE /api/custom-charts/<id>` | Borra el PNG y actualiza `index.json` |

### 8.2 Ejecución de notebooks en sandbox

La funcionalidad más sofisticada del admin es **ejecutar notebooks del analista desde el navegador**:

```python
# 1. Se convierte .ipynb a .py con jupyter nbconvert
subprocess.run(['jupyter', 'nbconvert', '--to', 'script', src, '--output-dir', tmpdir])

# 2. Se inyecta un wrapper que captura todos los plt.figure() generados
wrapper = f"""
import matplotlib; matplotlib.use('Agg')
{script_code}
for fig in plt.get_fignums():
    plt.figure(fig).savefig(CUSTOM_CHARTS_DIR / cid + '.png', dpi=120)
"""

# 3. Se ejecuta en subprocess con timeout de 120s
result = subprocess.run(['python3', '_wrapper.py'], capture_output=True, timeout=120)
```

Los gráficos generados se registran en `static/custom_charts/index.json` y aparecen automáticamente en el dashboard.

---
<a id='9'></a>
## 9. Arquitectura final de la aplicación

### 9.1 Diagrama de componentes

```
┌─────────────────────────────────────────────────────────────────┐
│                     BROWSER (Cliente)                           │
│  hipotecas.html      index.html       admin.html               │
│  [3-step advisor]    [dashboard]      [upload panel]           │
│  [Noa chat]          [Chart.js]       [chart sandbox]          │
└──────────────┬──────────────────────────────────────────────────┘
               │ HTTP/JSON
┌──────────────▼──────────────────────────────────────────────────┐
│                   FLASK app.py (:8000)                          │
│                                                                 │
│  /api/hipoteca-context   →  Macro + corr heatmap               │
│  /api/hipoteca-recomendacion → LinearReg + APRC + DTI + taxes  │
│  /api/noa-chat           →  Gemini 2.5 Flash (contextual)      │
│  /api/chat               →  Gemini 2.5 Flash (general)         │
│  /api/data               →  Aggregated dashboard data          │
│  /admin/upload-chart     →  nbconvert + subprocess sandbox     │
└──────────┬──────────────────────────────────────────────────────┘
           │
    ┌──────┴─────────────────────────────────────┐
    │  DATA LAYER                                │
    │  dataset_final_tfm.csv   (df en memoria)   │
    │  tipos_hipotecas.csv     (df_tipos)         │
    │  ITP/AJD constants       (dicts Python)     │
    │  custom_charts/index.json                  │
    └──────┬─────────────────────────────────────┘
           │
    ┌──────┴──────────────┐
    │  EXTERNAL APIs      │
    │  Google Gemini API  │
    └─────────────────────┘
```

### 9.2 Flujo del Asesor Inteligente (paso a paso)

```
PASO 1: Usuario selecciona año
   → GET /api/hipoteca-context?year=N
   → Macro KPIs (Euribor, APRC, Diferencial, IPC, Paro, Precio_m2)
   → Heatmap matplotlib (base64) + time series (tipos CSV)

PASO 2: Usuario elige hipoteca fija o variable
   → Solo UI, ninguna llamada al backend

PASO 3: Usuario introduce perfil (ingresos, ahorros, deudas, edad, provincia, precio)
   → POST /api/hipoteca-recomendacion
   → Lookup CCAA desde provincia
   → Tipos reales desde tipos_hipotecas.csv
   → Cálculo impuestos (ITP/IVA + AJD + notaría + registro)
   → max_financing_pct por CCAA y edad (bono joven)
   → Cuota mensual (amortización francesa)
   → DTI = (cuota + otras_deudas) / income
   → LinearRegression: contribuciones por variable
   → Decision: approved | analyst | denied

RESULTADO: Panel con tipo ofrecido + badge joven + impuestos + contribuciones
   → Noa pre-cargada con el contexto completo del cliente
   → POST /api/noa-chat (con contexto embebido en el prompt)
```

### 9.3 Resumen de tecnologías

| Capa | Tecnología |
|---|---|
| Backend | Python 3.9 · Flask |
| ML | scikit-learn (LinearRegression + StandardScaler) |
| Data | pandas · numpy |
| Visualización backend | matplotlib · seaborn (imágenes Base64 para la web) |
| Visualización frontend | Chart.js |
| IA generativa | Google Gemini 2.5 Flash |
| Frontend | HTML5 · CSS3 · JavaScript (Vanilla) |
| Diseño | Réplica adaptada del design system de CaixaBank |

## Pasos Finales de la Web: Noa, Proyecciones Futuras y Migración SDK (03-Mar-2026)

### 1. Asistente Noa (Dos Perfiles)
- **Noa Cliente** (`/api/noa-chat`): Gemini con prompt de asesora hipotecaria. Recibe el contexto íntegro del resultado (tipo, euribor, APRC, cuota, DTI, CCAA, etc.) para responder preguntas del cliente sobre su hipoteca.
- **Noa Strategic** (`/api/noa-strategic`): Visible solo en modo Gestor. Gemini con system prompt enfocado al gestor bancario: argumentario de conversión, explicación técnica del ML, análisis de rentabilidad para el banco.

### 2. Selector de Años Futuros (2025-2030) por Año Independiente
- Cada año tiene **proyecciones macroeconómicas diferenciadas** que se pasan al modelo ML como input.
- El modelo se entrena con todo el histórico hasta 2025 y predice sobre los valores proyectados.
- **Euribor** y **APRC** son valores distintos (Euribor ≠ APRC) gracias a la función `_get_tipos_proyectados()`, que mantiene el spread histórico separado.

### 3. Panel de Transparencia ML (años futuros)
- Cuando se selecciona un año ≥ 2025, la página muestra un panel con fondo ámbar que detalla:
  - Nº de trimestres históricos usados para entrenar el modelo
  - Cómo se proyectaron las variables macro (factor exponencial por año)
  - Métricas de cross-validación temporal (TimeSeriesSplit R² ± std, RMSE)
  - Importancias relativas de cada variable (feature_importances_)
  - Aviso de que la predicción es orientativa

### 4. Migración SDK Gemini (google.generativeai → google.genai)
- Instalado `google-genai` (versión 1.47.0)
- El import principal pasó de `import google.generativeai as genai` al nuevo `from google import genai`
- Se creó un wrapper `_ModelWrapper` que mantiene la API `.generate_content()` interna mientras el SDK de abajo usa `client.models.generate_content()`

### 5. Arquitectura y Estabilidad (resumen)
- `app.run(debug=True, port=8000, threaded=False)` — evita deadlocks de macOS con extensiones C de sklearn
- Todos los imports de sklearn y genai en scope global
- El endpoint `/api/hipoteca-recomendacion` devuelve `ml_transparency`, `is_future` y los campos ML para el frontend


## Actualizaciones y Mejoras del Dashboard y Simulador

Se han realizado varias tareas de mantenimiento, mejora de la experiencia de usuario (UX) e implementación de nuevas capacidades en la plataforma web:

1. **Gestión de Gráficos Personalizados:**
   - Se añadió un botón en el Dashboard Analítico (`index.html`) para eliminar gráficos personalizados que han sido inyectados desde el panel de administración.
   - La lógica está respaldada por el endpoint `/api/custom-charts/<id>` empleando el método DELETE en el backend (`app.py`).

2. **Filtro Específico por Año (Chart.js):**
   - Se incrustó un nuevo selector de año de forma individual sobre la tarjeta del gráfico de **Precio Medio Histórico por Provincia (€/m²)**.
   - Permite aislar la vista de este gráfico sin afectar al resto de filtros globales de línea temporal, actualizando su título dinámicamente.

3. **Nuevo Gráfico Dual Macro (Paro vs IPC):**
   - Se añadió un tercer KPI visual al panel principal de análisis: la correlación entre la Tasa de Paro y el IPC.
   - Construido como un gráfico mixto en Chart.js (barras para Paro y una línea superpuesta en eje Y secundario para IPC) para estudiar los factores macroeconómicos subyacentes.

4. **Correcciones de Layout HTML y CSS:**
   - Se parcheó el generador de plantillas literales de JavaScript en `hipotecas.html` para evitar la generación de etiquetas mal fomadas (ej., `< div class="info-box" >` debido a espacios intermedios) que corrompían el Flexbox de resultados.


## Sección: Prompts de Inteligencia Artificial (IA)

La plataforma cuenta con 3 asistentes basados en grandes modelos de lenguaje (gemini-2.5-flash) impulsados por Google GenAI. Se ha optado por dividir y especializar cada asistente con un contexto (*System Prompt*) diferente para asegurar que las directrices de CaixaBank se cumplan y la utilidad esté dirigida a la audiencia adecuada (Cliente Inmobiliario vs Gestor / Directivo).

### 1. Asistente IA (TFM) - Perfil Directivo
Este es el chatbot principal ubicado en el globo flotante de la página de inicio. Está orientado a responder dudas de negocio a nivel ejecutivo. 
*   **Capacidad Clave:** Conexión a Internet mediante *Google Search Grounding*. Puede consultar internet en vivo para obtener los últimos beneficios del banco, noticias de actualidad u otra info.
*   **Filtro Anti-Alucinación:** Tiene prohibido responder a preguntas "Fuera de Contexto" (OOC - *Out of Context*).

**Prompt del Sistema Utilizado:**
```python
prompt = f"""
Eres tu un Asistente Ejecutivo de Reporting para Directivos de CaixaBank.
Tu objetivo es responder preguntas sobre el mercado inmobiliario, macroeconomía, rentabilidad y beneficios o métricas de CaixaBank o del sector bancario.
Usa la herramienta de búsqueda en Google (cuando te pregunten cosas que no sepas de memoria o recientes) para buscar información en internet sobre beneficios de CaixaBank, noticias bancarias o datos macroeconómicos actuales.
Cruza la información de internet con los datos de nuestro dataset interno cuando sea útil.
IMPORTANTE: Si el usuario te pregunta cosas fuera de contexto (OOC - Out of Context) que no tengan que ver con macroeconomía, bancos, CaixaBank, hipotecas o el mercado inmobiliario, DEBES negarte a responder educadamente.

Datos CSV internos: {relevant}
Precio medio m2 histórico (toda la muestra): {df['Precio_m2'].mean():.2f} EUR.
Nota: Euribor, Spread_Hipotecas, tasa_ahorro e IPC son indicadores nacionales.

Pregunta del directivo: "{user_msg}"
"""
```


### 2. NOA Asesor Inmobiliario - Perfil Cliente / Usuario Final
Este bot interactúa en la página de resultados de viabilidad hipotecaria (`hipotecas.html`). Está diseñado para simular un trato comercial, cálido pero profesional, ayudando al cliente a entender la oferta.
*   **Capacidad Clave:** Acceso al DTI (Debt-to-Income), ahorros y explicación básica del tipo concedido, intentando persuadir al cliente.

**Prompt del Sistema Utilizado:**
```python
prompt = f"""
Eres NOA, Asesora Inmobiliaria de CaixaBank.
Tienes que hablarle a un cliente al que le acabamos de calcular un tipo de hipoteca.
Sé amable, persuasiva y muy clara.
Datos de la operacion:
Tipo dado: {data['differential']:.3f}% (Variable) o {data['fixed_rate']}% (Fijo).
Ahorros netos tras impuestos: {data['effective_savings']} EUR.
DTI real: {data['dti_ratio']:.1f}%

Habla de tú, explica por qué es una buena oportunidad o qué debe mejorar si va justo.
Mensaje del cliente: {user_msg}
"""
```


### 3. NOA Strategic - Perfil Gestor Interno
Este es un chatbot oculto (`manager-only`) que solo ve el gestor de CaixaBank (alerta amarilla). Analiza internamente el desglose de la regresión matemática.
*   **Capacidad Clave:** Lenguaje altamente técnico. Conoce las tripas del modelo (Gradient Boosting o Regresión Lineal) y cruza variables crudas.

**Prompt del Sistema Utilizado:**
```python
prompt = f"""
Eres NOA Strategic, asistente técnico interno para gestores de CaixaBank.
Hablas con un colega gestor. Sé directo y muy técnico.
Datos de la operación (ML engine):
- Rentabilidad extraída: {data.get('differential', 0):.3f}
- Razón (Macro features): {data.get('explanation', 'N/A')}

El objetivo del gestor es entender el riesgo real y cómo defender este tipo ante el cliente.
Dile qué variable (ej. IPC, Paro) está impactando más en este cálculo específico.
Mensaje del gestor: {user_msg}
"""
```


## Mejoras en Experiencia de Usuario del Chatbot (TFM)

Para dotar al Asistente Ejecutivo de un acabado más pulido e inteligente se han implementado las siguientes mejoras técnicas de front y back:

1.  **Estado "Escribiendo" y Cancelación (AbortController):**
    *   Se ha añadido un indicador visual dinámico en `script.js`: *"AI Asistente está escribiendo..."* mientras espera la respuesta de la API.
    *   Un botón rojo de **Cancelar** acompaña a este estado, vinculado a la API web nativa `AbortController`, lo que permite cancelar una promesa (`fetch`) asíncrona en curso ahorrando recursos del servidor.

2.  **Parseo de Markdown a HTML Frontend:**
    *   El modelo de lenguaje a menudo responde formateando elementos con sintaxis de Markdown (`**negrita**` o `*cursiva*`).
    *   Para procesarlo en la web, en `app.py` se ha reemplazado esto por las correspondientes etiquetas `<b>` e `<i>` usando Regex (`re.sub`). El frontend por su parte usa `innerHTML` para reflejar estas modificaciones estructurales y respetar los `<br>`.

3.  **Fuentes Verificadas (Grounding Metadata):**
    *   Habilitada la extracción de metadatos de Google Search Grounding en Gemini.
    *   El script en Python revisa iterativamente `response.candidates[0].grounding_metadata.grounding_chunks` para buscar `chunk.web.uri` y los compila en una lista HTML `<ul>` denominada **"Fuentes consultadas"**, inyectada dinámicamente al directivo para contrastar veracidad de la información macroeconómica que suministra.


### 🛠️ Corrección Crítica del Motor ML: Proyecciones a Futuro (2025-2030)

Se detectó un fallo crítico (HTTP Error 500) en la previsualización del simulador de hipotecas y en el renderizado de la interfaz, que impedía a los usuarios simular años futuros (2025-2030).

**Origen del Bug:**
Cuando el simulador interceptaba desde el frontend que el año elegido era >= 2025, el backend extraía un subconjunto *DataFrame* local. Sin embargo, los valores promedios agregados para variables macroeconómicas como el *Producto Interior Bruto per cápita (PIB_pc)* estaban vacíos (`NaN`) a partir de cierto trimestre de 2025 en el dataset de validación maestro de TFM. Esto provocaba que Python inyectase respuestas de la forma `{ "pib_pc": NaN }`, corrompiendo el parser JSON nativo de la interfaz web, lo cual bloqueaba todos los desplegables en cascada que dependían de la API `/api/hipoteca-context`.

**Resolución Aplicada en Python `app.py`:**
1.  **Manejo de Valores Nulos (NaN Handling):** Se modificaron los endpoints (`/api/hipoteca-context` y `/api/simulate`) para envolver las transformaciones matriciales con `.fillna(0)` a nivel de agrupación macro (e.g. `ydf[['IPC', 'Tasa_Paro', 'PIB_pc', 'tasa_ahorro']].mean().fillna(0)`), resolviendo los conflictos JSON `NaN`. A nivel provincial, si `Precio_m2` es inválido `(pd.isna)` también se rellena para no quebrar el código.
2.  **Sintonía Fija en Tipos Diferenciales:** A petición del usuario y por mayor fiabilidad, se procedió a hacer un "Rollback" del vector maestro y restauramos la fuente de regresión de tipos de interés apuntando el `TIPOS_PATH` de vuelta al archivo estático `static/tipos_hipotecas.csv`. Este mismo archivo, que contiene los ratios correctos de `Euribor` e `IPC` histórico hasta finales de 2025 (`Q4`), sigue proveyendo la realidad económica y contextual al Prompt general del *AI Assistant (Directivo)* y al motor algorítmico global de *NOA*.


### 📊 Transparencia Algorítmica en Predicciones Futuras (2026-2030)

Como requerimiento de transparencia hacia los gestores y clientes simulando años futuros, se ha instrumentado una Nota Metodológica de UI:

1.  **Exposición de Flag Predictivo (Backend):** 
    En `app.py`, el endpoint de contexto Macro (`/api/hipoteca-context`) expone en el payload JSON un boleano `is_future: True` si el año de simulación sobrepasa los registros oficiales constatados (hasta 2025, de cara a 2026-2030).
2.  **Inyección Condicional de Documentación (Frontend):** 
    Este disparador habilita a través de JavaScript un panel informativo destacado previo a las cuadrículas de datos sobre cómo se han formulado los cálculos en los indicadores:
    *   **Euribor Oficial:** Extrapolado con descensos escalonados anuales acumulativos del -8% en previsión de desescalada (políticas de estímulo del tipo BCE).
    *   **Diferencial Préstamo:** Ensanchamiento del +1% contrapuesto ante rebajas de tipos.
    *   **Inflación (IPC):** Descenso estabilizador del -10% anual hacia el target base económico.
    *   **Precio Vivienda:** Conservación de inercia positiva (+2% promedio M^2). 

Esto elimina la "Caja Negra" en las visualizaciones de Machine Learning de la herramienta de hipotecas de CaixaBank.


### 🧠 Actualización a Econometría Real en el Simulador WEB (2026-2030)

Respondiendo a requisitos técnicos de rigor matemático, se ha descartado el uso de diferenciales porcentuales fijos (ej. -8% anual) para la proyección del Euribor e indicadores económicos en los años sin datos (2026-2030).

**Implementación en Backend (`app.py`):**
1.  **Modelo SARIMAX (Tipos de Interés):** Se integró la librería `statsmodels` (usada previamente en el core del análisis de Data Science del TFM) para ejecutar inferencia en tiempo real de la serie de tiempo. Específicamente, se levanta un modelo temporal **SARIMAX (1,1,1)** en memoria que aprende de todo el CSV histórico del Banco de España y proyecta a 5 años (`steps=5`) el Euribor y el Diferencial, ajustando la nomenclatura a una escala aditiva de **puntos porcentuales**.
2.  **Regresión Lineal (Extrapolación Macro):** Tratándose de la inflación (IPC) y el Paro, se implementó una Regresión Lineal automática como estimador robusto, protegiendo las inferencias extremas con recortes de rangos funcionales (`np.clip`).
3.  **Caché en Memoria:** Computar *Machine Learning* temporal en cada click del usuario web causaría latencia. Se instauró una caché global (`_MACRO_FORECASTS`) que pre-entrena estos modelos una sola vez por ciclo de vida de la aplicación.


### 📈 Estabilización Predictiva: Modelo SARIMAX (1,1,0) para Paro e IPC y KPI de PIB

Al aplicar modelos asintóticos de Regresión Lineal pura, se detectaron disfuncionalidades en los enlaces pre-2025 frente al horizonte virtual de 2026+: saltos irreales de hasta un +5% en el parámetro predictivo del Tasa de Paro no justificado matemáticamente y atribuible al re-escalado tendencial.

**Refactorización Matemática en `app.py`:**
1.  **Modelo Autoforecast Estacionario:** Todas las predicciones de factores Macro (IPC, Tasa Paro, PIB per cápita, Precio m2) abandonan la regresión simple y convergen ahora en **SARIMAX (1,1,0)**. Al añadir el anclaje de un grado de diferenciación (`d=1`), aseguramos por fuerza bruta inferencial que el salto analítico de Diciembre de 2025 al primer trimestre de 2026 parta exactamente del último cuantil empírico, encadenando un "paseo aleatorio anclado" suavizado por un término autorregresivo.
2.  **Tracking del Crecimiento Nacional:** Se incrustaron medidores del crecimiento subyacente del PIB Interanual real y proyectado. Para las llamadas del Simulador a `/api/hipoteca-context`, el servidor compara iterativamente los tensores del estimador versus el año anterior (-1 año) para proveer un indicador UI al cliente final (Ej. +2.10%).
